# Task
Compare the performance of LightGBM and XGBoost models for predicting survival on the Titanic dataset, using `Titanic_train.csv` for training and `Titanic_test.csv` for testing, after performing data loading, exploration, and preprocessing steps.

In [1]:
import pandas as pd

train_df = pd.read_csv('/content/Titanic_train.csv')
test_df = pd.read_csv('/content/Titanic_test.csv')

print("Train DataFrame loaded successfully.")
print("Test DataFrame loaded successfully.")

Train DataFrame loaded successfully.
Test DataFrame loaded successfully.


## Explore Data

### Subtask:
Perform initial exploratory data analysis (EDA) to understand the dataset structure, identify missing values, and examine data types. This will help inform preprocessing steps.


**Reasoning**:
To begin the exploratory data analysis, I will display the first few rows, shape, data types, and summary statistics, and check for missing values for both the training and testing datasets as per the instructions.



In [2]:
print("--- Train DataFrame Head ---")
print(train_df.head())
print("\n--- Test DataFrame Head ---")
print(test_df.head())

print("\n--- Train DataFrame Shape ---")
print(train_df.shape)
print("\n--- Test DataFrame Shape ---")
print(test_df.shape)

print("\n--- Train DataFrame Info ---")
train_df.info()
print("\n--- Test DataFrame Info ---")
test_df.info()

print("\n--- Train DataFrame Descriptive Statistics ---")
print(train_df.describe())
print("\n--- Test DataFrame Descriptive Statistics ---")
print(test_df.describe())

print("\n--- Train DataFrame Missing Values ---")
print(train_df.isnull().sum())
print("\n--- Test DataFrame Missing Values ---")
print(test_df.isnull().sum())

--- Train DataFrame Head ---
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450

## Preprocess Data

### Subtask:
Clean and preprocess the data. This includes handling missing values (e.g., imputation), encoding categorical features (e.g., one-hot encoding), and dropping irrelevant columns.


**Reasoning**:
I will define a function `preprocess_data` to encapsulate all the specified data cleaning and preprocessing steps. This function will handle missing values for 'Age', 'Embarked', and 'Fare' by imputing with their respective medians/modes within the input DataFrame, drop irrelevant columns, convert 'Sex' to numerical, and one-hot encode 'Embarked'. Then, I will apply this function to both `train_df` and `test_df` and display their head and info to verify the preprocessing.



In [3]:
def preprocess_data(df):
    # Fill missing 'Age' values with the median
    df['Age'].fillna(df['Age'].median(), inplace=True)

    # Fill missing 'Embarked' values with the mode
    # Note: Mode might return multiple values if there's a tie, so take the first one.
    if 'Embarked' in df.columns and df['Embarked'].isnull().any():
        df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

    # Fill missing 'Fare' values with the median
    if 'Fare' in df.columns and df['Fare'].isnull().any():
        df['Fare'].fillna(df['Fare'].median(), inplace=True)

    # Drop irrelevant columns
    columns_to_drop = ['Cabin', 'Name', 'Ticket', 'PassengerId']
    df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

    # Convert 'Sex' column to numerical representation (0 for female, 1 for male)
    df['Sex'] = df['Sex'].map({'female': 0, 'male': 1})

    # Perform one-hot encoding on the 'Embarked' column
    if 'Embarked' in df.columns:
        df = pd.get_dummies(df, columns=['Embarked'], drop_first=True, dtype=int)

    return df

# Apply the preprocessing function to both DataFrames
train_df = preprocess_data(train_df.copy())
test_df = preprocess_data(test_df.copy())

# Display the first few rows and info of the preprocessed DataFrames
print("--- Preprocessed Train DataFrame Head ---")
print(train_df.head())
print("\n--- Preprocessed Train DataFrame Info ---")
train_df.info()

print("\n--- Preprocessed Test DataFrame Head ---")
print(test_df.head())
print("\n--- Preprocessed Test DataFrame Info ---")
test_df.info()

--- Preprocessed Train DataFrame Head ---
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked_Q  Embarked_S
0         0       3    1  22.0      1      0   7.2500           0           1
1         1       1    0  38.0      1      0  71.2833           0           0
2         1       3    0  26.0      0      0   7.9250           0           1
3         1       1    0  35.0      1      0  53.1000           0           1
4         0       3    1  35.0      0      0   8.0500           0           1

--- Preprocessed Train DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Sex         891 non-null    int64  
 3   Age         891 non-null    float64
 4   SibSp       891 non-null    int64  
 5   Parch       891 non-null    int64  
 6   Fare      

/tmp/ipython-input-1526859262.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipython-input-1526859262.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

**Reasoning**:
The previous preprocessing step successfully cleaned the data, so now I need to prepare the data for model training by separating features from the target variable for the training set and ensuring feature consistency between the training and test sets.



In [4]:
X_train = train_df.drop('Survived', axis=1)
y_train = train_df['Survived']
X_test = test_df.copy()

# Ensure columns are consistent between training and testing sets
# This is crucial for models like LightGBM/XGBoost
common_cols = list(set(X_train.columns) & set(X_test.columns))
X_train = X_train[common_cols]
X_test = X_test[common_cols]

# Reorder columns to be the same, if necessary (though intersection keeps order somewhat consistent)
X_test = X_test[X_train.columns]

print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)

Shape of X_train: (891, 8)
Shape of y_train: (891,)
Shape of X_test: (418, 8)


## Train LightGBM Model

### Subtask:
Train a LightGBM classifier on the preprocessed training data (`X_train`, `y_train`).

**Reasoning**:
To train the LightGBM classifier, I need to import the `LGBMClassifier` model from the `lightgbm` library, initialize it, and then fit it to the `X_train` and `y_train` datasets.



In [5]:
import lightgbm as lgb

# Initialize the LightGBM Classifier
lightgbm_model = lgb.LGBMClassifier(random_state=42)

# Train the model on the training data
lightgbm_model.fit(X_train, y_train)

print("LightGBM model trained successfully.")

[LightGBM] [Info] Number of positive: 342, number of negative: 549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000344 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 222
[LightGBM] [Info] Number of data points in the train set: 891, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383838 -> initscore=-0.473288
[LightGBM] [Info] Start training from score -0.473288
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM model trained successfully.


## Train XGBoost Model

### Subtask:
Train an XGBoost classifier on the preprocessed training data (`X_train`, `y_train`).

**Reasoning**:
To train the XGBoost classifier, I need to import the `XGBClassifier` model from the `xgboost` library, initialize it, and then fit it to the `X_train` and `y_train` datasets.



In [6]:
import xgboost as xgb

# Initialize the XGBoost Classifier
xgb_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)

# Train the model on the training data
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

XGBoost model trained successfully.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [17:46:35] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


**Reasoning**:
The previous code generated a UserWarning because the `use_label_encoder` parameter in `XGBClassifier` is deprecated and no longer used. I will remove this parameter from the `XGBClassifier` initialization to eliminate the warning, as it does not affect the model's functionality.



In [7]:
import xgboost as xgb

# Initialize the XGBoost Classifier
xgb_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42)

# Train the model on the training data
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

XGBoost model trained successfully.


## Evaluate Models

### Subtask:
Make predictions on the test dataset (`X_test`) using both the trained LightGBM and XGBoost models. Then evaluate their performance using appropriate metrics.

**Reasoning**:
To evaluate the LightGBM model, I will first use the trained `lightgbm_model` to make predictions on the `X_test` dataset. These predictions will then be used for performance metric calculation.



In [8]:
lightgbm_predictions = lightgbm_model.predict(X_test)
print("LightGBM predictions on X_test generated.")

LightGBM predictions on X_test generated.


**Reasoning**:
With LightGBM predictions generated, I will now use the trained `xgb_model` to make predictions on the `X_test` dataset to complete the prediction step for both models.



In [9]:
xgb_predictions = xgb_model.predict(X_test)
print("XGBoost predictions on X_test generated.")

XGBoost predictions on X_test generated.


## Evaluate LightGBM

### Subtask:
Evaluate the performance of the LightGBM model using appropriate metrics such as accuracy, precision, recall, F1-score, and a confusion matrix.


**Reasoning**:
To understand the distribution of LightGBM's predictions without actual labels, I will calculate and display the total number of predictions and the counts of 'survived' (1) and 'not survived' (0) classifications.



In [10]:
import numpy as np

# 1. Print the total number of predictions
total_predictions_lightgbm = len(lightgbm_predictions)
print(f"Total LightGBM predictions: {total_predictions_lightgbm}")

# 2. Calculate and print the count of 'survived' (1) and 'not survived' (0) predictions
lightgbm_prediction_counts = np.bincount(lightgbm_predictions.astype(int))

# Ensure both 0 and 1 are accounted for, even if one is not present
not_survived_count = lightgbm_prediction_counts[0] if len(lightgbm_prediction_counts) > 0 else 0
survived_count = lightgbm_prediction_counts[1] if len(lightgbm_prediction_counts) > 1 else 0

print(f"LightGBM predicted 'Not Survived' (0) count: {not_survived_count}")
print(f"LightGBM predicted 'Survived' (1) count: {survived_count}")

Total LightGBM predictions: 418
LightGBM predicted 'Not Survived' (0) count: 256
LightGBM predicted 'Survived' (1) count: 162


**Reasoning**:
To understand the distribution of XGBoost's predictions without actual labels, I will calculate and display the total number of predictions and the counts of 'survived' (1) and 'not survived' (0) classifications.



In [11]:
import numpy as np

# 1. Print the total number of predictions
total_predictions_xgb = len(xgb_predictions)
print(f"Total XGBoost predictions: {total_predictions_xgb}")

# 2. Calculate and print the count of 'survived' (1) and 'not survived' (0) predictions
xgb_prediction_counts = np.bincount(xgb_predictions.astype(int))

# Ensure both 0 and 1 are accounted for, even if one is not present
not_survived_count_xgb = xgb_prediction_counts[0] if len(xgb_prediction_counts) > 0 else 0
survived_count_xgb = xgb_prediction_counts[1] if len(xgb_prediction_counts) > 1 else 0

print(f"XGBoost predicted 'Not Survived' (0) count: {not_survived_count_xgb}")
print(f"XGBoost predicted 'Survived' (1) count: {survived_count_xgb}")

Total XGBoost predictions: 418
XGBoost predicted 'Not Survived' (0) count: 246
XGBoost predicted 'Survived' (1) count: 172


## Compare Models

### Subtask:
Compare the evaluation results of both LightGBM and XGBoost to determine which algorithm performed better on the Titanic dataset, considering the limitations due to the absence of true labels for the test set.


## Summary:

### Q&A

**How do LightGBM and XGBoost models compare in predicting survival on the Titanic dataset, considering the limitations due to the absence of true labels for the test set?**

Due to the absence of true labels for the test set (`y_test`), a direct comparison of the models' performance using standard metrics like accuracy, precision, or F1-score is not possible. However, we can compare the distribution of their predictions:

*   **LightGBM** predicted 162 individuals would survive and 256 would not survive out of 418 test passengers.
*   **XGBoost** predicted 172 individuals would survive and 246 would not survive out of 418 test passengers.

XGBoost predicted a slightly higher number of survivors (172 vs. 162) and consequently a lower number of non-survivors compared to LightGBM.

### Data Analysis Key Findings

*   The training dataset (`train_df`) contained 891 rows and 12 columns, while the test dataset (`test_df`) had 418 rows and 11 columns.
*   Missing values were identified:
    *   `train_df`: 177 missing values in 'Age', 687 in 'Cabin', and 2 in 'Embarked'.
    *   `test_df`: 86 missing values in 'Age', 1 in 'Fare', and 327 in 'Cabin'.
*   Data preprocessing involved:
    *   Imputing missing 'Age' and 'Fare' with their respective medians, and 'Embarked' with its mode.
    *   Dropping columns like 'Cabin', 'Name', 'Ticket', and 'PassengerId' due to irrelevance or high missingness.
    *   Encoding 'Sex' into numerical format (0 for female, 1 for male) and one-hot encoding 'Embarked'.
    *   The preprocessed training features (`X_train`) and test features (`X_test`) both had 8 columns.
*   Both LightGBM and XGBoost classifier models were successfully trained on the preprocessed training data.
*   Predictions were generated for the test set for both models:
    *   LightGBM predicted 162 'Survived' (1) and 256 'Not Survived' (0).
    *   XGBoost predicted 172 'Survived' (1) and 246 'Not Survived' (0).
*   A comprehensive evaluation using metrics such as accuracy, precision, or recall could not be performed due to the unavailability of true labels for the test set (`y_test`).

### Insights or Next Steps

*   To perform a robust comparison and select the best model, it is crucial to obtain true labels for the test set or implement a cross-validation strategy on the training data to estimate performance more reliably.
*   Given the slight differences in prediction distributions, further analysis (e.g., feature importance from each model, or examining specific cases where predictions differ) could provide insights into their distinct decision-making processes, even without true test labels.
